# INC-003 Sleep Silver and Gold Review

This notebook inspects an existing pipeline run against the INC-003 contract (`specs/INC-003.md`): typed sleep silver tables, twelve sleep quality rules, and the six sleep columns of `gold_clinical_daily`. The supplied database is opened read-only. The idempotency section reruns the pipeline against an isolated copy. Raw inputs are read only.

Generate the run first with `python -m wearable_pipeline run --data-dir data --output-dir output/runs/inc003`. Set `INC003_RUN_DIR` if the output directory differs, `INC003_DATA_DIR` to override `data/`, and `INC003_PREVIOUS_RUN_DIR` (default `output/runs/inc002-01`, an INC-002-01 run) to compare the pre-existing gold columns.

## Review Sections
1. Environment, paths and display helpers.
2. Row counts: gold, silver sleep sessions and stages.
3. Silver grain, key uniqueness and schemas.
4. Sleep rule counts against the profiling findings (S-01..S-05).
5. Gold sleep columns: coverage, null patterns, metric ranges and the efficiency range.
   Includes the gold schema and example records per situation (sections 5b, 5c).
6. S-04 recomputed with the stage-based formula.
7. Existing gold columns versus a previous run (optional).
8. Rerun on an isolated copy: identical tables and report; raw data unchanged.
9. Acceptance summary with PASS/FAIL/NOT RUN.

Aggregates are displayed, plus a few explicitly selected example gold and silver records (sleep metrics only); reject raw-value payloads are not printed.

In [34]:
import hashlib
import html
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "wearable_pipeline").is_dir()
)
DATA_DIR = Path(os.environ.get("INC003_DATA_DIR", ROOT / "data")).resolve()
RUN_DIR = Path(os.environ.get("INC003_RUN_DIR", ROOT / "output" / "runs" / "inc003")).resolve()
PREVIOUS_RUN_DIR = Path(
    os.environ.get("INC003_PREVIOUS_RUN_DIR", ROOT / "output" / "runs" / "inc002-01")
).resolve()
DATABASE_PATH = RUN_DIR / "wearable.duckdb"
REPORT_PATH = RUN_DIR / "quality" / "report.md"

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f"DuckDB file not found: {DATABASE_PATH}. Set INC003_RUN_DIR.")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Raw data directory not found: {DATA_DIR}")

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<h4>{html.escape(title)}</h4><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


report_text = REPORT_PATH.read_text(encoding="utf-8")
run_id = next(
    (line.split(": ", 1)[1] for line in report_text.splitlines() if line.startswith("Run id: ")),
    None,
)
print(f"Database: {DATABASE_PATH}")
print(f"Data directory: {DATA_DIR}")
print(f"Run ID: {run_id}")
print(f"DuckDB version: {duckdb.__version__}")


Database: /home/michaelmadeira/wearable-health-data-pipeline/output/runs/inc003/wearable.duckdb
Data directory: /home/michaelmadeira/wearable-health-data-pipeline/data
Run ID: a826c8397cf5ad8e
DuckDB version: 1.5.6


In [23]:
# Section 2: row counts against the profiled dataset.
expected_counts = {
    "gold_clinical_daily": 1400,
    "silver_sleep_sessions": 1400,
    "silver_sleep_stages": 7000,
    "quality_results": 19,
}
observed_counts = {
    name: connection.execute(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]
    for name in expected_counts
}
render_table(
    "Expected versus observed row counts",
    ["table", "expected", "observed", "status"],
    [(n, e, observed_counts[n], "PASS" if observed_counts[n] == e else "FAIL") for n, e in expected_counts.items()],
)
acceptance["Row counts match the profiled dataset"] = all(
    observed_counts[n] == e for n, e in expected_counts.items()
)

table,expected,observed,status
gold_clinical_daily,1400,1400,PASS
silver_sleep_sessions,1400,1400,PASS
silver_sleep_stages,7000,7000,PASS
quality_results,19,19,PASS


In [24]:
# Section 3: schemas, grain and key uniqueness of the sleep silver tables.
show_query(
    "Silver sleep schemas",
    """SELECT table_name, ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name IN ('silver_sleep_sessions', 'silver_sleep_stages')
       ORDER BY table_name, ordinal_position""",
)
session_key = connection.execute(
    """SELECT COUNT(*) - COUNT(DISTINCT (participant_id, local_date)),
              COUNT(*) FILTER (WHERE local_date IS NULL),
              COUNT(*) FILTER (WHERE is_duplicate_key OR is_orphan OR is_out_of_window)
       FROM silver_sleep_sessions"""
).fetchone()
stage_key = connection.execute(
    """SELECT COUNT(*) - COUNT(DISTINCT (participant_id, local_date, stage_index)),
              COUNT(*) FILTER (WHERE stage IS NULL OR duration_min IS NULL)
       FROM silver_sleep_stages"""
).fetchone()
render_table(
    "Grain and key checks",
    ["check", "observed", "status"],
    [
        ("Duplicate session keys", session_key[0], "PASS" if session_key[0] == 0 else "FAIL"),
        ("Sessions with NULL local_date", session_key[1], "PASS" if session_key[1] == 0 else "FAIL"),
        ("Sessions flagged duplicate/orphan/out of window", session_key[2], "INFO"),
        ("Duplicate stage keys", stage_key[0], "PASS" if stage_key[0] == 0 else "FAIL"),
        ("Stages with NULL name or duration", stage_key[1], "INFO"),
    ],
)
acceptance["Sleep silver keys are unique and dates parse"] = (
    session_key[0] == 0 and session_key[1] == 0 and stage_key[0] == 0
)

table_name,ordinal_position,column_name,data_type
silver_sleep_sessions,1,participant_id,VARCHAR
silver_sleep_sessions,2,local_date,DATE
silver_sleep_sessions,3,sleep_onset,TIMESTAMP
silver_sleep_sessions,4,sleep_end,TIMESTAMP
silver_sleep_sessions,5,supplied_efficiency_pct,DOUBLE
silver_sleep_sessions,6,restlessness,DOUBLE
silver_sleep_sessions,7,is_duplicate_key,BOOLEAN
silver_sleep_sessions,8,is_orphan,BOOLEAN
silver_sleep_sessions,9,is_out_of_window,BOOLEAN
silver_sleep_sessions,10,quality_flags,VARCHAR


check,observed,status
Duplicate session keys,0,PASS
Sessions with NULL local_date,0,PASS
Sessions flagged duplicate/orphan/out of window,0,INFO
Duplicate stage keys,0,PASS
Stages with NULL name or duration,0,INFO


In [25]:
# Section 4: sleep rule counts versus the profiling findings (S-04 is recorded, not compared).
show_query(
    "Sleep quality results",
    """SELECT rule_id, handling, severity, checked_count, failed_count
       FROM quality_results WHERE source = 'sleep' ORDER BY rule_id""",
)
failed = dict(connection.execute(
    "SELECT rule_id, failed_count FROM quality_results WHERE source = 'sleep'"
).fetchall())
profiled = {
    "SLEEP_STAGE_DURATION_MISMATCH": 4,
    "SLEEP_SPAN_OUTLIER": 3,
    "SLEEP_FINAL_STAGE_MISALIGNED": 3,
    "SLEEP_ONSET_NOT_MINUTE_ALIGNED": 1379,
}
render_table(
    "Profiled defect counts (S-01, S-02, S-03, S-05)",
    ["rule_id", "profiled", "observed", "status"],
    [(r, e, failed.get(r), "PASS" if failed.get(r) == e else "FAIL") for r, e in profiled.items()],
)
print("SLEEP_EFFICIENCY_MISMATCH (S-04) observed:", failed.get("SLEEP_EFFICIENCY_MISMATCH"))
unobserved = [
    "SLEEP_ORPHAN_PARTICIPANT", "SLEEP_DUPLICATE_PARTICIPANT_DAY", "SLEEP_DATE_UNPARSEABLE",
    "SLEEP_DATE_OUT_OF_WINDOW", "SLEEP_SESSION_FIELD_INVALID", "SLEEP_STAGE_INVALID",
    "SLEEP_STAGE_GAP_OR_OVERLAP",
]
render_table(
    "Structural-defect rules (none observed in profiling)",
    ["rule_id", "observed"],
    [(r, failed.get(r)) for r in unobserved],
)
acceptance["Profiled S-01, S-02, S-03 and S-05 counts reproduced"] = all(
    failed.get(r) == e for r, e in profiled.items()
)

rule_id,handling,severity,checked_count,failed_count
SLEEP_DATE_OUT_OF_WINDOW,exclude_from_output,medium,1400,0
SLEEP_DATE_UNPARSEABLE,exclude_from_output,medium,1400,0
SLEEP_DUPLICATE_PARTICIPANT_DAY,withhold_field,high,1400,0
SLEEP_EFFICIENCY_MISMATCH,flag,medium,1400,1393
SLEEP_FINAL_STAGE_MISALIGNED,flag,medium,1400,3
SLEEP_ONSET_NOT_MINUTE_ALIGNED,flag,medium,1400,1379
SLEEP_ORPHAN_PARTICIPANT,exclude_from_output,medium,1400,0
SLEEP_SESSION_FIELD_INVALID,withhold_field,medium,1400,0
SLEEP_SPAN_OUTLIER,flag,medium,1400,3
SLEEP_STAGE_DURATION_MISMATCH,flag,medium,1400,4


rule_id,profiled,observed,status
SLEEP_STAGE_DURATION_MISMATCH,4,4,PASS
SLEEP_SPAN_OUTLIER,3,3,PASS
SLEEP_FINAL_STAGE_MISALIGNED,3,3,PASS
SLEEP_ONSET_NOT_MINUTE_ALIGNED,1379,1379,PASS


SLEEP_EFFICIENCY_MISMATCH (S-04) observed: 1393


rule_id,observed
SLEEP_ORPHAN_PARTICIPANT,0
SLEEP_DUPLICATE_PARTICIPANT_DAY,0
SLEEP_DATE_UNPARSEABLE,0
SLEEP_DATE_OUT_OF_WINDOW,0
SLEEP_SESSION_FIELD_INVALID,0
SLEEP_STAGE_INVALID,0
SLEEP_STAGE_GAP_OR_OVERLAP,0


In [26]:
# Section 5: gold sleep columns.
show_query(
    "Gold sleep coverage",
    """SELECT COUNT(*) AS gold_rows,
              COUNT(*) FILTER (WHERE sleep_present) AS sleep_present,
              COUNT(*) FILTER (WHERE sleep_session_count > 1) AS multi_session_days,
              COUNT(*) FILTER (WHERE total_sleep_min IS NULL) AS null_total,
              COUNT(*) FILTER (WHERE deep_sleep_pct IS NULL) AS null_deep_pct,
              COUNT(*) FILTER (WHERE sleep_efficiency_pct IS NULL) AS null_efficiency,
              COUNT(*) FILTER (WHERE sleep_quality_flags IS NOT NULL) AS flagged_days
       FROM gold_clinical_daily""",
)
show_query(
    "Gold sleep metric ranges (efficiency is stage-based; compare with profiling findings section 7)",
    """SELECT MIN(total_sleep_min) AS min_total, MAX(total_sleep_min) AS max_total,
              ROUND(MIN(deep_sleep_pct), 2) AS min_deep_pct, ROUND(MAX(deep_sleep_pct), 2) AS max_deep_pct,
              ROUND(MIN(sleep_efficiency_pct), 2) AS min_efficiency,
              ROUND(MAX(sleep_efficiency_pct), 2) AS max_efficiency
       FROM gold_clinical_daily""",
)
show_query(
    "Flag combinations on gold days",
    """SELECT sleep_quality_flags, COUNT(*) AS days
       FROM gold_clinical_daily WHERE sleep_quality_flags IS NOT NULL
       GROUP BY 1 ORDER BY days DESC, 1 LIMIT 15""",
)
bounds = connection.execute(
    """SELECT COUNT(*) FILTER (WHERE deep_sleep_pct < 0 OR deep_sleep_pct > 100
                              OR sleep_efficiency_pct < 0 OR sleep_efficiency_pct > 100
                              OR total_sleep_min < 0),
              COUNT(*) FILTER (WHERE sleep_present <> (sleep_session_count > 0))
       FROM gold_clinical_daily"""
).fetchone()
render_table(
    "Gold invariants",
    ["check", "violations", "status"],
    [
        ("Metrics outside their valid ranges", bounds[0], "PASS" if bounds[0] == 0 else "FAIL"),
        ("sleep_present disagrees with sleep_session_count", bounds[1], "PASS" if bounds[1] == 0 else "FAIL"),
    ],
)
acceptance["Gold sleep invariants hold"] = bounds == (0, 0)

gold_rows,sleep_present,multi_session_days,null_total,null_deep_pct,null_efficiency,flagged_days
1400,1400,0,0,0,0,1400


min_total,max_total,min_deep_pct,max_deep_pct,min_efficiency,max_efficiency
168,700,2.99,71.77,74.57,94.37


sleep_quality_flags,days
"SLEEP_EFFICIENCY_MISMATCH,SLEEP_ONSET_NOT_MINUTE_ALIGNED",1365
SLEEP_EFFICIENCY_MISMATCH,21
SLEEP_ONSET_NOT_MINUTE_ALIGNED,7
"SLEEP_EFFICIENCY_MISMATCH,SLEEP_ONSET_NOT_MINUTE_ALIGNED,SLEEP_STAGE_DURATION_MISMATCH",4
"SLEEP_EFFICIENCY_MISMATCH,SLEEP_FINAL_STAGE_MISALIGNED,SLEEP_ONSET_NOT_MINUTE_ALIGNED,SLEEP_SPAN_OUTLIER",3


check,violations,status
Metrics outside their valid ranges,0,PASS
sleep_present disagrees with sleep_session_count,0,PASS


In [27]:
# Section 5b: gold_clinical_daily schema and primary key.
show_query(
    "Gold schema (the six sleep columns are appended after the ten survey columns)",
    """SELECT ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position""",
)
show_query(
    "Gold key constraints",
    """SELECT constraint_type, constraint_column_names
       FROM duckdb_constraints() WHERE table_name = 'gold_clinical_daily'""",
)
gold_columns = [row[0] for row in connection.execute(
    "SELECT column_name FROM information_schema.columns "
    "WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position").fetchall()]
acceptance["Gold has 16 columns ending with the six sleep columns"] = (
    len(gold_columns) == 16
    and gold_columns[-6:] == ["total_sleep_min", "deep_sleep_pct", "sleep_efficiency_pct",
                               "sleep_session_count", "sleep_present", "sleep_quality_flags"]
)

ordinal_position,column_name,data_type
1,participant_id,VARCHAR
2,local_date,DATE
3,fatigue_score,INTEGER
4,stress_score,INTEGER
5,readiness_score,INTEGER
6,sleep_quality_score,INTEGER
7,survey_present,BOOLEAN
8,survey_row_count,INTEGER
9,survey_valid_score_count,INTEGER
10,survey_quality_flags,VARCHAR


constraint_type,constraint_column_names
PRIMARY KEY,"['participant_id', 'local_date']"
NOT NULL,['participant_id']
NOT NULL,['local_date']


In [28]:
# Section 5c: example gold records (one deterministic example per situation, sleep columns only plus the key).
sleep_columns = ("participant_id, local_date, total_sleep_min, ROUND(deep_sleep_pct, 2) AS deep_sleep_pct, "
                 "ROUND(sleep_efficiency_pct, 2) AS sleep_efficiency_pct, sleep_session_count, "
                 "sleep_present, sleep_quality_flags")
situations = [
    ("Clean sleep day", "sleep_present AND sleep_quality_flags IS NULL"),
    ("Day with only the onset-alignment flag (S-05)",
     "sleep_quality_flags = 'SLEEP_ONSET_NOT_MINUTE_ALIGNED'"),
    ("Day with a stage duration mismatch (S-01)", "sleep_quality_flags LIKE '%SLEEP_STAGE_DURATION_MISMATCH%'"),
    ("Day with a span outlier (S-02)", "sleep_quality_flags LIKE '%SLEEP_SPAN_OUTLIER%'"),
    ("Day with a misaligned final stage (S-03)", "sleep_quality_flags LIKE '%SLEEP_FINAL_STAGE_MISALIGNED%'"),
    ("Day with an efficiency mismatch (S-04)", "sleep_quality_flags LIKE '%SLEEP_EFFICIENCY_MISMATCH%'"),
    ("Day with several sessions", "sleep_session_count > 1"),
    ("Day without sleep", "NOT sleep_present"),
]
for title, condition in situations:
    cursor = connection.execute(
        f"SELECT {sleep_columns} FROM gold_clinical_daily WHERE {condition} "
        "ORDER BY participant_id, local_date LIMIT 1"
    )
    rows = cursor.fetchall()
    render_table(title, [item[0] for item in cursor.description],
                 rows or [("no such day in this run",) + ("",) * (len(cursor.description) - 1)])
show_query(
    "Same example day across layers: gold row, its silver session and stage count",
    """SELECT g.participant_id, g.local_date, g.total_sleep_min, s.sleep_onset, s.sleep_end,
              s.supplied_efficiency_pct, ROUND(g.sleep_efficiency_pct, 2) AS derived_efficiency_pct,
              (SELECT COUNT(*) FROM silver_sleep_stages t
               WHERE t.participant_id = s.participant_id AND t.local_date = s.local_date) AS stages
       FROM gold_clinical_daily g
       JOIN silver_sleep_sessions s USING (participant_id, local_date)
       WHERE g.sleep_present AND NOT s.is_duplicate_key
       ORDER BY g.participant_id, g.local_date LIMIT 3""",
)

participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
no such day in this run,,,,,,,


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
P003,2026-01-23,421,16.39,88.82,1,True,SLEEP_ONSET_NOT_MINUTE_ALIGNED


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
P007,2026-01-21,523,71.13,88.34,1,True,"SLEEP_EFFICIENCY_MISMATCH,SLEEP_ONSET_NOT_MINUTE_ALIGNED,SLEEP_STAGE_DURATION_MISMATCH"


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
P026,2026-01-23,379,10.82,86.53,1,True,"SLEEP_EFFICIENCY_MISMATCH,SLEEP_FINAL_STAGE_MISALIGNED,SLEEP_ONSET_NOT_MINUTE_ALIGNED,SLEEP_SPAN_OUTLIER"


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
P026,2026-01-23,379,10.82,86.53,1,True,"SLEEP_EFFICIENCY_MISMATCH,SLEEP_FINAL_STAGE_MISALIGNED,SLEEP_ONSET_NOT_MINUTE_ALIGNED,SLEEP_SPAN_OUTLIER"


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
P001,2026-01-08,490,18.78,87.34,1,True,"SLEEP_EFFICIENCY_MISMATCH,SLEEP_ONSET_NOT_MINUTE_ALIGNED"


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
no such day in this run,,,,,,,


participant_id,local_date,total_sleep_min,deep_sleep_pct,sleep_efficiency_pct,sleep_session_count,sleep_present,sleep_quality_flags
no such day in this run,,,,,,,


participant_id,local_date,total_sleep_min,sleep_onset,sleep_end,supplied_efficiency_pct,derived_efficiency_pct,stages
P001,2026-01-08,490,2026-01-07 22:54:29,2026-01-08 07:04:29,100.0,87.34,5
P001,2026-01-09,391,2026-01-08 23:29:00,2026-01-09 06:00:00,100.0,88.46,5
P001,2026-01-10,366,2026-01-09 22:55:49,2026-01-10 05:01:49,100.0,90.37,5


In [35]:
# Section 5d: example gold records with every column (survey and sleep together).
full_situations = [
    ("Sleep day with only the onset-alignment flag", "sleep_quality_flags = 'SLEEP_ONSET_NOT_MINUTE_ALIGNED'"),
    ("Sleep day with an efficiency mismatch", "sleep_quality_flags LIKE '%SLEEP_EFFICIENCY_MISMATCH%'"),
    ("Day without a survey", "NOT survey_present"),
    ("Day without sleep", "NOT sleep_present"),
]
for title, condition in full_situations:
    cursor = connection.execute(
        f"SELECT * FROM gold_clinical_daily WHERE {condition} ORDER BY participant_id, local_date LIMIT 2"
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    # Transposed so all 16 columns fit on screen: one row per column, one value column per example day.
    render_table(
        title,
        ["column"] + [f"example {n}" for n in range(1, len(rows) + 1)] if rows else ["column"],
        [(name, *(row[i] for row in rows)) for i, name in enumerate(columns)] if rows else [("no such day in this run",)],
    )


column,example 1,example 2
participant_id,P003,P004
local_date,2026-01-23,2026-01-28
fatigue_score,2,2
stress_score,3,3
readiness_score,5,5
sleep_quality_score,4,3
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


column,example 1,example 2
participant_id,P001,P001
local_date,2026-01-08,2026-01-09
fatigue_score,3,3
stress_score,3,3
readiness_score,8,6
sleep_quality_score,3,3
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


column
no such day in this run


column
no such day in this run


In [29]:
# Section 6: S-04 recomputed from silver with the stage-based formula (independent of the pipeline code path).
recomputed = connection.execute(
    """WITH per_session AS (
           SELECT s.participant_id, s.local_date, s.supplied_efficiency_pct,
                  SUM(t.duration_min) FILTER (WHERE t.stage <> 'awake') * 100.0
                      / NULLIF(SUM(t.duration_min), 0) AS derived
           FROM silver_sleep_sessions s
           JOIN silver_sleep_stages t USING (participant_id, local_date)
           WHERE NOT s.is_duplicate_key
           GROUP BY ALL)
       SELECT COUNT(*) FILTER (WHERE ABS(supplied_efficiency_pct - derived) > 1),
              ROUND(MIN(derived), 2), ROUND(MAX(derived), 2),
              ROUND(MAX(ABS(supplied_efficiency_pct - derived)), 3)
       FROM per_session"""
).fetchone()
render_table(
    "S-04 recomputed (supplied vs derived efficiency, tolerance 1 point)",
    ["sessions over tolerance", "min derived", "max derived", "max abs difference"],
    [recomputed],
)
print("Pipeline SLEEP_EFFICIENCY_MISMATCH failed_count:", failed.get("SLEEP_EFFICIENCY_MISMATCH"))
acceptance["S-04 recomputation agrees with the pipeline count"] = (
    recomputed[0] == failed.get("SLEEP_EFFICIENCY_MISMATCH")
)

sessions over tolerance,min derived,max derived,max abs difference
1393,74.57,94.37,39.381


Pipeline SLEEP_EFFICIENCY_MISMATCH failed_count: 1393


In [30]:
# Section 7: pre-existing gold columns versus a previous (INC-002-01) run, when supplied.
existing_columns = (
    "participant_id, local_date, fatigue_score, stress_score, readiness_score, "
    "sleep_quality_score, survey_present, survey_row_count, survey_valid_score_count, "
    "survey_quality_flags"
)
if PREVIOUS_RUN_DIR:
    previous_db = Path(PREVIOUS_RUN_DIR).resolve() / "wearable.duckdb"
    previous = duckdb.connect(str(previous_db), read_only=True)
    try:
        before = previous.execute(f"SELECT {existing_columns} FROM gold_clinical_daily ORDER BY ALL").fetchall()
    finally:
        previous.close()
    after = connection.execute(f"SELECT {existing_columns} FROM gold_clinical_daily ORDER BY ALL").fetchall()
    identical = before == after
    render_table(
        "Existing gold columns versus previous run",
        ["rows before", "rows after", "identical", "status"],
        [(len(before), len(after), identical, "PASS" if identical else "FAIL")],
    )
    acceptance["Existing gold columns identical to the previous run"] = identical
else:
    print("INC003_PREVIOUS_RUN_DIR not set: comparison NOT RUN.")
    acceptance["Existing gold columns identical to the previous run"] = None

rows before,rows after,identical,status
1400,1400,True,PASS


In [31]:
# Section 8: rerun on an isolated copy and check raw-data immutability against the run manifest.
connection.close()
sys.path.insert(0, str(ROOT / "src"))
from wearable_pipeline.pipeline import run_pipeline

table_names = [
    "raw_manifest", "silver_participants", "silver_wellness_survey", "silver_sleep_sessions",
    "silver_sleep_stages", "quality_results", "quality_rejects", "gold_clinical_daily",
]
rerun_comparisons = []
with tempfile.TemporaryDirectory(prefix="inc003-idempotency-review-") as temporary_root:
    temporary_root = Path(temporary_root)
    baseline_dir = temporary_root / "baseline"
    rerun_dir = temporary_root / "rerun"
    for directory in (baseline_dir, rerun_dir):
        (directory / "quality").mkdir(parents=True)
        shutil.copy2(DATABASE_PATH, directory / "wearable.duckdb")
        shutil.copy2(REPORT_PATH, directory / "quality" / "report.md")

    rerun_result = run_pipeline(DATA_DIR, rerun_dir)
    before_db = duckdb.connect(str(baseline_dir / "wearable.duckdb"), read_only=True)
    after_db = duckdb.connect(str(rerun_dir / "wearable.duckdb"), read_only=True)
    try:
        for table_name in table_names:
            before_rows = before_db.execute(f'SELECT * FROM "{table_name}" ORDER BY ALL').fetchall()
            after_rows = after_db.execute(f'SELECT * FROM "{table_name}" ORDER BY ALL').fetchall()
            rerun_comparisons.append((table_name, len(before_rows), before_rows == after_rows))
    finally:
        before_db.close()
        after_db.close()
    report_identical = (baseline_dir / "quality" / "report.md").read_bytes() == (
        rerun_dir / "quality" / "report.md"
    ).read_bytes()
    run_id_identical = rerun_result.run_id == run_id

render_table(
    "Same-input rerun comparison",
    ["table", "rows before", "identical after rerun", "status"],
    [(*row, "PASS" if row[2] else "FAIL") for row in rerun_comparisons],
)
render_table(
    "Rerun determinism",
    ["check", "result", "status"],
    [
        ("Run ID unchanged", run_id_identical, "PASS" if run_id_identical else "FAIL"),
        ("Quality report bytes unchanged", report_identical, "PASS" if report_identical else "FAIL"),
    ],
)
acceptance["Rerun preserves every table's logical contents"] = all(row[2] for row in rerun_comparisons)
acceptance["Rerun preserves the deterministic run id and report"] = run_id_identical and report_identical

check_db = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = {
    row[0]: row[1]
    for row in check_db.execute("SELECT relative_path, sha256 FROM raw_manifest").fetchall()
}
check_db.close()
hash_mismatches = [
    path for path, digest in manifest.items()
    if hashlib.sha256((DATA_DIR / path).read_bytes()).hexdigest() != digest
]
print("Raw files differing from the run manifest SHA-256:", hash_mismatches[:5] or "none")
acceptance["Raw files match the run manifest hashes"] = not hash_mismatches

table,rows before,identical after rerun,status
raw_manifest,154,True,PASS
silver_participants,50,True,PASS
silver_wellness_survey,1414,True,PASS
silver_sleep_sessions,1400,True,PASS
silver_sleep_stages,7000,True,PASS
quality_results,19,True,PASS
quality_rejects,2796,True,PASS
gold_clinical_daily,1400,True,PASS


check,result,status
Run ID unchanged,True,PASS
Quality report bytes unchanged,True,PASS


Raw files differing from the run manifest SHA-256: none


In [32]:
# Section 9: acceptance summary.
rows = [
    (name, "PASS" if result is True else "FAIL" if result is False else "NOT RUN")
    for name, result in acceptance.items()
]
rows.append(("Changed-input updates affect exactly one gold row", "NOT RUN here; covered by the pipeline tests"))
render_table("INC-003 review summary", ["acceptance check", "status"], rows)
print("Review is observational; only the isolated temporary database copy was rerun.")

acceptance check,status
Row counts match the profiled dataset,PASS
Sleep silver keys are unique and dates parse,PASS
"Profiled S-01, S-02, S-03 and S-05 counts reproduced",PASS
Gold sleep invariants hold,PASS
Gold has 16 columns ending with the six sleep columns,PASS
S-04 recomputation agrees with the pipeline count,PASS
Existing gold columns identical to the previous run,PASS
Rerun preserves every table's logical contents,PASS
Rerun preserves the deterministic run id and report,PASS
Raw files match the run manifest hashes,PASS


Review is observational; only the isolated temporary database copy was rerun.
